# Ensemble Methods


In [2]:
import pandas as pd

import numpy as np

## Choose your own Regression

**Working together with your partner**, choose an interesting data set to fit *regression* models for, that is, a data set with a *quantitative* target variable and several feature variables. Could be a data set that: we have used in class, we have used on an assignment, a data set from your project, you used in another course, or you just find interesting! You'll work together with your partner to develop *regression* models to predict the *quantitative* target (so you need to decide on the data set and target variable together with your partner).

After you have decided on the data set and target variable:

- One partner should fit a linear regression model
- One partner should fit a k-nearest neighbors regression model

If you're using an example from earlier in the class, you can use a model you have already come up with or you can try something new. Note: you do NOT have to agree on the features included in your models.

After each partner has their model, **work together to combine your code and create two more models**:

- An ensemble model that uses **voting**
- An ensemble model that uses **stacking**

For each of the four models (linear regression, knn regression, voting ensemble, stacking ensemble)

1. Fit the model using the training data
1. Use the fitted model to predict the target for a few observations of your choice, and compare the predictions
1. Estimate the test RMSE using cross-validation, and compare the results

Which model seems to work best?


### Data and models

We use the local Ames housing data to predict sale price in dollars. All four models use the same four numerical features so their predictions can be compared directly. The one row with a missing basement area is removed. The scaling step is fitted separately within each training fold during cross-validation.


In [3]:
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import VotingRegressor, StackingRegressor

ames = pd.read_csv("AmesHousing.txt", sep="\t")
regression_features = ["Gr Liv Area", "Overall Qual", "Year Built", "Total Bsmt SF"]
regression_data = ames[regression_features + ["SalePrice"]].dropna()

X_reg = regression_data[regression_features]
y_reg = regression_data["SalePrice"]
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

linear_reg = make_pipeline(StandardScaler(), LinearRegression())
knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10))
voting_reg = VotingRegressor([
    ("linear", make_pipeline(StandardScaler(), LinearRegression())),
    ("knn", make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10)))
])
stacking_reg = StackingRegressor(
    estimators=[
        ("linear", make_pipeline(StandardScaler(), LinearRegression())),
        ("knn", make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10)))
    ],
    final_estimator=LinearRegression(),
    cv=5
)

regression_models = {
    "Linear regression": linear_reg,
    "KNN regression": knn_reg,
    "Voting regression": voting_reg,
    "Stacking regression": stacking_reg
}

for model in regression_models.values():
    model.fit(X_reg_train, y_reg_train)


### Predictions for five held-out houses

These houses were excluded from model fitting. The actual price is included for comparison; the model ranking below comes from cross-validation.


In [4]:
houses = X_reg_test.head(5)
regression_predictions = pd.DataFrame({"Actual sale price": y_reg_test.loc[houses.index]})
for name, model in regression_models.items():
    regression_predictions[name] = model.predict(houses)

regression_predictions.round(0)


,Actual sale price,Linear regression,KNN regression,Voting regression,Stacking regression
1473,208500,217177.0,197655.0,207416.0,202456.0
2179,171500,153369.0,166891.0,160130.0,165056.0
368,260000,235450.0,228461.0,231955.0,232044.0
2615,183000,201960.0,176355.0,189158.0,181553.0
1819,132500,119919.0,139665.0,129792.0,135925.0


### Cross-validated RMSE

Five-fold cross-validation estimates prediction error on houses that were held out from each fold. Lower RMSE is better. Cross-validation uses only the training portion, leaving the five examples above as a separate comparison.


In [5]:
regression_rmse = {}
for name, model in regression_models.items():
    scores = cross_validate(
        model, X_reg_train, y_reg_train, cv=5,
        scoring="neg_root_mean_squared_error"
    )
    regression_rmse[name] = -scores["test_score"].mean()

regression_results = pd.Series(regression_rmse, name="CV RMSE ($)").sort_values()
print(regression_results.round(0))
print("\nLowest estimated RMSE:", regression_results.index[0])


Stacking regression    30368.0
KNN regression         30549.0
Voting regression      31150.0
Linear regression      35004.0
Name: CV RMSE ($), dtype: float64

Lowest estimated RMSE: Stacking regression


The model printed with the lowest cross-validated RMSE seems to work best for predicting sale price. Voting averages the two base predictions, while stacking trains a linear regression to combine them. An ensemble is not guaranteed to improve on its strongest individual model.


## Voting in Classification

Suppose you have three models for predicting a categorical target:

- k-nearest neighbors
- logistic regression
- decision tree

How might you apply voting to create an ensemble model? Discuss conceptually with your partner. Create a concrete example to illustrate your method. Hint: There are two basic approaches to voting in classification; can you think of both?

Note: You can assume a binary target, but the general idea of voting applies to multiclass problems. However, if any of the candidate *models* are not naturally multiclass (like logistic regression) then you have to consider things like OvR and OvO as we have discussed.

### Two kinds of voting

**Hard voting:** Each model chooses a class. The ensemble chooses the class receiving the most votes. For example, if KNN predicts "yes," logistic regression predicts "yes," and the decision tree predicts "no," the ensemble predicts "yes" by a 2-to-1 vote.

**Soft voting:** Each model supplies a probability for each class. Average the three probabilities for each class, then choose the class with the highest average. For example, if the probabilities of "yes" are 0.60 (KNN), 0.70 (logistic regression), and 0.25 (tree), their average is about 0.52. The average probability of "no" is about 0.48, so the ensemble predicts "yes." This illustrates that soft voting uses confidence as well as class labels. All three models must supply class probabilities, and their class labels must refer to the same target.


## Choose your own Classification

**Working together with your partner**, choose an interesting data set to fit *classification* models for, that is, a data set with a *categorical* target variable and several feature variables. Could be a data set that: we have used in class, we have used on an assignment, a data set from your project, you used in another course, or you just find interesting! You'll work together with your partner to develop *classification* models to predict the *categorical* target (so you need to decide on the data set and target variable together with your partner).

After you have decided on the data set and target variable:

- One partner should fit a k-nearest neighbors classification model
- One partner should fit another classification model like decision tree or logistic regression

If you're using an example from earlier in the class, you can use a model you have already come up with or you can try something new. Note: you do NOT have to agree on the features included in your models.

After each partner has their model, **work together to combine your code and create an ensemble model that uses *voting***.

For each of the three models (knn classification, other classification, voting ensemble)

1. Fit the model using the training data
1. Use the fitted model to predict the target for a few observations of your choice, and compare the predictions
1. Estimate the test macro precision, recall, and F1 score using cross-validation, and compare the results

Which model seems to work best?

**Hint:** [`VotingClassifier`](https://scikit-learn.org/1.5/modules/generated/sklearn.ensemble.VotingClassifier.html) works almost identically to `VotingRegressor`.


### Data and models

We use the local loans data to predict homeownership (MORTGAGE, RENT, or OWN) from four numerical loan application fields. KNN uses scaling because it measures distances; the logistic regression pipeline uses the same scaling. We remove rows missing a selected field and use a stratified split to preserve the class proportions.


In [6]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier

loans = pd.read_csv("loans_full_schema.csv")
classification_features = [
    "annual_income", "debt_to_income", "loan_amount", "total_credit_limit"
]
classification_data = loans[classification_features + ["homeownership"]].dropna()

X_class = classification_data[classification_features]
y_class = classification_data["homeownership"]
X_class_train, X_class_test, y_class_train, y_class_test = train_test_split(
    X_class, y_class, test_size=0.25, stratify=y_class, random_state=42
)

knn_class = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))
log_class = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
voting_class = VotingClassifier(
    estimators=[
        ("knn", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))),
        ("logistic", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))
    ],
    voting="soft"
)

classification_models = {
    "KNN classification": knn_class,
    "Logistic regression": log_class,
    "Soft voting": voting_class
}

for model in classification_models.values():
    model.fit(X_class_train, y_class_train)


### Predictions for five held-out loans

Soft voting averages the class probabilities from KNN and logistic regression. With two models, hard voting can tie when they disagree; soft voting provides a useful comparison of their probabilities.


In [7]:
loan_examples = X_class_test.head(5)
classification_predictions = pd.DataFrame({
    "Actual homeownership": y_class_test.loc[loan_examples.index]
})
for name, model in classification_models.items():
    classification_predictions[name] = model.predict(loan_examples)

classification_predictions


,Actual homeownership,KNN classification,Logistic regression,Soft voting
1215,MORTGAGE,RENT,RENT,RENT
4109,RENT,RENT,RENT,RENT
9027,MORTGAGE,RENT,RENT,RENT
6460,MORTGAGE,MORTGAGE,MORTGAGE,MORTGAGE
5047,MORTGAGE,MORTGAGE,MORTGAGE,MORTGAGE


### Cross-validated macro scores

Five-fold cross-validation estimates precision, recall, and F1 on unseen folds of the training portion. Macro scoring gives each homeownership class equal weight. Higher scores are better.


In [8]:
classification_scores = {}
for name, model in classification_models.items():
    scores = cross_validate(
        model, X_class_train, y_class_train, cv=5,
        scoring=["precision_macro", "recall_macro", "f1_macro"]
    )
    classification_scores[name] = {
        "Macro precision": scores["test_precision_macro"].mean(),
        "Macro recall": scores["test_recall_macro"].mean(),
        "Macro F1": scores["test_f1_macro"].mean()
    }

classification_results = pd.DataFrame.from_dict(classification_scores, orient="index")
classification_results = classification_results.sort_values("Macro F1", ascending=False)
print(classification_results.round(3))
print("\nHighest estimated macro F1:", classification_results.index[0])


C:\Users\jaxle\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\jaxle\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\jaxle\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\jaxle\anaconda3\Lib

                     Macro precision  Macro recall  Macro F1
KNN classification             0.540         0.549     0.514
Soft voting                    0.482         0.556     0.512
Logistic regression            0.480         0.554     0.510

Highest estimated macro F1: KNN classification


C:\Users\jaxle\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


The model printed with the highest cross-validated macro F1 seems to work best overall for these three classes. The precision and recall columns show whether that choice also performs well on those separate criteria.
